# Demo 10. Jacobi and Gauss-Seidel Iteration

**Standard 7** (iterative methods), Week 6.

Iterative solvers approximate the solution of $Ax=b$ by repeated cheap updates rather than a direct factorization. This demo implements Jacobi and Gauss-Seidel, measures their convergence, and connects the observed rate to the spectral radius of the iteration matrix.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Convergence criterion

Split $A = D + L + U$. Each method has the fixed-point form $x_{k+1} = M x_k + c$ with iteration matrix

$$ M_{\text{Jacobi}} = -D^{-1}(L+U), \qquad M_{\text{GS}} = -(D+L)^{-1}U. $$

The error satisfies $e_{k+1} = M e_k$, so $e_k = M^k e_0$. The iteration converges for every $x_0$ if and only if the spectral radius satisfies $\rho(M) < 1$, and the asymptotic error reduction per step approaches $\rho(M)$. Strict diagonal dominance is a sufficient condition for both methods to converge.

In [ ]:
# ---------------------------------------------------------------------------
# Splitting-based iterations for A x = b
# ---------------------------------------------------------------------------
# Write A = D + L + U (diagonal, strict lower, strict upper). Each method solves
# an easy system per step:
#
#   Jacobi        : D x_{k+1} = b - (L + U) x_k
#   Gauss-Seidel  : (D + L) x_{k+1} = b - U x_k   (uses updated entries at once)
#
# Both take the form x_{k+1} = M x_k + c. The method converges for any starting
# vector if and only if the spectral radius rho(M) < 1, and the asymptotic rate
# is governed by rho(M).

def jacobi(A, b, x0, n_iter):
    """Jacobi iteration. Returns the iterates and the per-step error proxy
    (residual norm ||A x_k - b||)."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    D = np.diag(np.diag(A))
    R = A - D                              # L + U
    Dinv = np.diag(1.0 / np.diag(A))
    x = x0.copy()
    res = [np.linalg.norm(A @ x - b)]
    for _ in range(n_iter):
        x = Dinv @ (b - R @ x)
        res.append(np.linalg.norm(A @ x - b))
    return x, np.array(res)

def gauss_seidel(A, b, x0, n_iter):
    """Gauss-Seidel iteration using forward substitution with (D + L)."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    n = len(b)
    L = np.tril(A)                         # D + strict lower
    U = A - L                              # strict upper
    x = x0.copy()
    res = [np.linalg.norm(A @ x - b)]
    for _ in range(n_iter):
        # solve (D + L) x_new = b - U x by forward substitution
        rhs = b - U @ x
        x_new = np.zeros(n)
        for i in range(n):
            x_new[i] = (rhs[i] - L[i, :i] @ x_new[:i]) / L[i, i]
        x = x_new
        res.append(np.linalg.norm(A @ x - b))
    return x, np.array(res)

def spectral_radius(M):
    """Return rho(M) = max |eigenvalue|."""
    return np.max(np.abs(np.linalg.eigvals(M)))

def iteration_matrices(A):
    """Return the Jacobi and Gauss-Seidel iteration matrices M_J, M_GS."""
    D = np.diag(np.diag(A)); R = A - D
    M_J = -np.linalg.solve(D, R)
    L = np.tril(A); U = A - L
    M_GS = -np.linalg.solve(L, U)
    return M_J, M_GS

In [ ]:
# ---------------------------------------------------------------------------
# A test matrix whose diagonal dominance we can tune
# ---------------------------------------------------------------------------
def make_matrix(n=20, dominance=2.0):
    """Symmetric matrix with adjustable diagonal dominance.

    Off-diagonals are -1 on the first sub/super-diagonal; the diagonal is
    `dominance` times the row's off-diagonal sum. Larger `dominance` gives a
    smaller spectral radius and faster convergence.
    """
    A = np.zeros((n, n))
    for i in range(n):
        if i > 0:     A[i, i-1] = -1.0
        if i < n-1:   A[i, i+1] = -1.0
    off = np.sum(np.abs(A), axis=1)
    np.fill_diagonal(A, dominance * np.maximum(off, 1.0))
    return A

In [ ]:
# ---------------------------------------------------------------------------
# Compare convergence and report the spectral radii
# ---------------------------------------------------------------------------
def show_iterations(n=20, dominance=2.0, n_iter=40):
    """Run both methods on the tunable matrix and plot residual vs iteration."""
    A = make_matrix(n, dominance)
    x_true = np.ones(n)
    b = A @ x_true
    x0 = np.zeros(n)

    _, res_j = jacobi(A, b, x0, n_iter)
    _, res_gs = gauss_seidel(A, b, x0, n_iter)
    M_J, M_GS = iteration_matrices(A)

    plt.figure()
    plt.semilogy(res_j, "r.-", label=f"Jacobi   (rho={spectral_radius(M_J):.3f})")
    plt.semilogy(res_gs, "g.-", label=f"Gauss-Seidel (rho={spectral_radius(M_GS):.3f})")
    plt.xlabel("iteration"); plt.ylabel("residual  ||A x - b||")
    plt.title(f"n={n}, diagonal dominance factor={dominance:.2f}")
    plt.legend(); plt.show()

    print(f"rho(Jacobi)       = {spectral_radius(M_J):.4f}")
    print(f"rho(Gauss-Seidel) = {spectral_radius(M_GS):.4f}")
    print("Both < 1 here, so both converge; the smaller rho converges faster.")

show_iterations(20, 2.0, 40)

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: reduce the diagonal dominance toward 1 and watch rho approach 1
# ---------------------------------------------------------------------------
interact(
    show_iterations,
    n=IntSlider(value=20, min=5, max=60, step=5, description="n"),
    dominance=FloatSlider(value=2.0, min=0.55, max=4.0, step=0.05, description="dominance"),
    n_iter=IntSlider(value=40, min=10, max=150, step=10, description="# iters"),
);

## Summary

- Jacobi and Gauss-Seidel replace factorization with repeated updates using the splitting $A=D+L+U$.
- Convergence for all starting vectors holds if and only if $\rho(M)<1$; the rate is set by $\rho(M)$.
- Gauss-Seidel uses updated components within each sweep and typically has the smaller spectral radius, hence faster convergence.
- Reducing the diagonal dominance toward 1 drives $\rho(M)$ toward 1 and the convergence slows.